# Market Basket Analysis with Apriori and FP-Growth

This notebook demonstrates how to perform market basket analysis using association rule mining algorithms. We'll explore transaction data, apply Apriori and FP-Growth algorithms to find frequent itemsets, and generate meaningful association rules.

## 1. Import Required Libraries

We need to import libraries for data manipulation, machine learning algorithms, and visualization.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. Load and Explore the Dataset

Let's load the market basket dataset and perform initial exploration to understand its structure.

In [ ]:
# Load the dataset
df = pd.read_csv('../backend/datasets/market_basket.csv')

# Display basic information
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Get unique items
all_items = set()
for col in df.columns[1:]:  # Skip TransactionID
    all_items.update(df[col].dropna().unique())

print(f"\nTotal unique items: {len(all_items)}")
print("Sample items:", list(all_items)[:10])

## 3. Data Preprocessing

We need to convert the transaction data into a format suitable for the association rule mining algorithms.

In [ ]:
# Preprocess the data
transactions = []
for _, row in df.iterrows():
    transaction = [item for item in row[1:] if pd.notna(item)]
    transactions.append(transaction)

print(f"Number of transactions: {len(transactions)}")
print("Sample transactions:")
for i, transaction in enumerate(transactions[:5]):
    print(f"Transaction {i+1}: {transaction}")

# Use TransactionEncoder to convert to one-hot encoded format
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

print(f"\nOne-hot encoded shape: {basket_df.shape}")
print("Columns (items):", list(basket_df.columns))

## 4. Apply Apriori Algorithm

The Apriori algorithm finds frequent itemsets by iteratively generating candidate itemsets and pruning those that don't meet the minimum support threshold.

In [ ]:
# Apply Apriori algorithm
min_support = 0.3  # Minimum support threshold
frequent_itemsets_apriori = apriori(basket_df, min_support=min_support, use_colnames=True)

print(f"Frequent itemsets found with Apriori (min_support={min_support}):")
print(frequent_itemsets_apriori)

# Sort by support
frequent_itemsets_apriori = frequent_itemsets_apriori.sort_values(by='support', ascending=False)
print("\nTop frequent itemsets:")
print(frequent_itemsets_apriori.head())

## 5. Apply FP-Growth Algorithm

FP-Growth is generally faster than Apriori, especially for large datasets, as it uses a tree structure to mine frequent itemsets.

In [ ]:
# Apply FP-Growth algorithm
frequent_itemsets_fpgrowth = fpgrowth(basket_df, min_support=min_support, use_colnames=True)

print(f"Frequent itemsets found with FP-Growth (min_support={min_support}):")
print(frequent_itemsets_fpgrowth)

# Sort by support
frequent_itemsets_fpgrowth = frequent_itemsets_fpgrowth.sort_values(by='support', ascending=False)
print("\nTop frequent itemsets:")
print(frequent_itemsets_fpgrowth.head())

# Verify both algorithms give same results
print("\nResults match:", frequent_itemsets_apriori.equals(frequent_itemsets_fpgrowth.sort_index()))

## 6. Generate Association Rules

From the frequent itemsets, we can generate association rules using different metrics like confidence, lift, and conviction.

In [ ]:
# Generate association rules using confidence metric
min_confidence = 0.6
rules = association_rules(frequent_itemsets_fpgrowth, metric="confidence", min_threshold=min_confidence)

print(f"Association rules generated (min_confidence={min_confidence}):")
print(rules)

# Sort by confidence
rules = rules.sort_values(by='confidence', ascending=False)
print("\nTop rules by confidence:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

## 7. Evaluate and Interpret Rules

Let's filter rules based on different criteria and interpret the results.

In [ ]:
# Filter rules with high lift (> 1.2)
high_lift_rules = rules[rules['lift'] > 1.2]

print("Rules with high lift (>1.2):")
print(high_lift_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

# Find rules with specific items
bread_rules = rules[rules['antecedents'].apply(lambda x: 'Bread' in str(x)) | 
                   rules['consequents'].apply(lambda x: 'Bread' in str(x))]

print("\nRules involving Bread:")
print(bread_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

# Interpret key insights
print("\nKey Insights:")
for _, rule in rules.iterrows():
    ant = list(rule['antecedents'])
    cons = list(rule['consequents'])
    conf = rule['confidence']
    lift = rule['lift']
    print(f"If customers buy {ant}, they are {conf:.2%} likely to buy {cons} (lift: {lift:.2f})")

## 8. Visualize Frequent Itemsets

Create visualizations to better understand the frequent itemsets and their relationships.

In [ ]:
# Visualize support of frequent itemsets
plt.figure(figsize=(10, 6))
top_items = frequent_itemsets_fpgrowth.nlargest(10, 'support')
bars = plt.bar(range(len(top_items)), top_items['support'])
plt.xlabel('Itemset')
plt.ylabel('Support')
plt.title('Top 10 Frequent Itemsets by Support')

# Add itemset labels
labels = [str(list(itemset)) for itemset in top_items['itemsets']]
plt.xticks(range(len(top_items)), labels, rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Visualize association rules as scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(rules['support'], rules['confidence'], alpha=0.5, s=rules['lift']*100)
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules: Support vs Confidence (bubble size = lift)')

# Add rule labels for top rules
for i, rule in rules.head(3).iterrows():
    ant = list(rule['antecedents'])
    cons = list(rule['consequents'])
    plt.annotate(f"{ant} -> {cons}", (rule['support'], rule['confidence']), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.grid(True, alpha=0.3)
plt.show()

# Heatmap of item co-occurrence (simplified)
item_support = basket_df.mean().sort_values(ascending=False)
top_10_items = item_support.head(10)

co_occurrence = basket_df[top_10_items.index].T.dot(basket_df[top_10_items.index])
np.fill_diagonal(co_occurrence.values, 0)  # Remove self-co-occurrence

plt.figure(figsize=(10, 8))
sns.heatmap(co_occurrence, annot=True, cmap='YlOrRd', fmt='.2f')
plt.title('Item Co-occurrence Matrix (Top 10 Items)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()